Environment Setup and Library Imports

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


Google Drive Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Local Unzipping of Datasets and Clean Path Detection

In [ ]:
train_zip = "/content/drive/MyDrive/Colab Notebooks/train.zip"
val_zip = "/content/drive/MyDrive/Colab Notebooks/valid.zip"
test_zip = "/content/drive/MyDrive/Colab Notebooks/test.zip"

os.makedirs("/content/train_local", exist_ok=True)
os.makedirs("/content/val_local", exist_ok=True)
os.makedirs("/content/test_local", exist_ok=True)

!unzip -q "{train_zip}" -d /content/train_local
!unzip -q "{val_zip}" -d /content/val_local
!unzip -q "{test_zip}" -d /content/test_local

def get_image_dir(base_path):
    for root, dirs, files in os.walk(base_path):
        if '__MACOSX' in root:
            continue
        valid_files = [f for f in files if not f.startswith('.') and f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if valid_files:
            return root
    return base_path

train_img_dir = get_image_dir("/content/train_local")
val_img_dir = get_image_dir("/content/val_local")
test_img_dir = get_image_dir("/content/test_local")

print(f"Train image dir: {train_img_dir}")
print(f"Val image dir:   {val_img_dir}")
print(f"Test image dir:  {test_img_dir}")

Train image dir: /content/train_local/train
Val image dir:   /content/val_local/valid
Test image dir:  /content/test_local/test


Dataframes Loading and CUI Vocabulary Creation

In [ ]:
train_csv_path = "/content/drive/MyDrive/Colab Notebooks/train_concepts.csv"
val_csv_path = "/content/drive/MyDrive/Colab Notebooks/valid_concepts.csv"
test_csv_path = "/content/drive/MyDrive/Colab Notebooks/test_concepts.csv"

train_df = pd.read_csv(train_csv_path)
val_df = pd.read_csv(val_csv_path)
test_df = pd.read_csv(test_csv_path)

def align_dataframe(df, img_dir):
    if not os.path.exists(img_dir):
        return pd.DataFrame(columns=df.columns)

    extracted_files = {f.lower() for f in os.listdir(img_dir) if not f.startswith('.')}

    def fix_filename(x):
        s = str(x).strip()
        if not any(s.lower().endswith(ext) for ext in ['.jpg', '.jpeg', '.png']):
            return f"{s}.jpg"
        return s

    df['filename'] = df['ID'].apply(fix_filename)
    filtered_df = df[df['filename'].str.lower().isin(extracted_files)].reset_index(drop=True)
    return filtered_df

train_df = align_dataframe(train_df, train_img_dir)
val_df = align_dataframe(val_df, val_img_dir)
test_df = align_dataframe(test_df, test_img_dir)

print(f"Dataset aligned: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

unique_cuis = set()
for cuis_string in train_df['CUIs'].dropna():
    cuis_list = [cui.strip() for cui in str(cuis_string).split(';')]
    unique_cuis.update(cuis_list)

sorted_cuis = sorted(list(unique_cuis))
cui_to_idx = {cui: idx for idx, cui in enumerate(sorted_cuis)}
idx_to_cui = {idx: cui for cui, idx in cui_to_idx.items()}

print(f"Total Unique CUI Concepts: {len(cui_to_idx)}")

Dataset aligned: Train=59958, Val=9904, Test=9927
Total Unique CUI Concepts: 1947


PyTorch Dataset Definition

In [ ]:
class ROCOv2ConceptDataset(Dataset):
    def __init__(self, df, img_dir, cui_to_idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.cui_to_idx = cui_to_idx
        self.transform = transform
        self.num_classes = len(cui_to_idx)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            next_idx = (idx + 1) % len(self.df)
            return self[next_idx]

        if self.transform:
            image = self.transform(image)

        target = torch.zeros(self.num_classes, dtype=torch.float32)
        if pd.notna(row['CUIs']):
            cuis = [cui.strip() for cui in str(row['CUIs']).split(';')]
            for cui in cuis:
                if cui in self.cui_to_idx:
                    target[self.cui_to_idx[cui]] = 1.0

        return image, target

Data Transformations and DataLoaders Setup

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ROCOv2ConceptDataset(train_df, train_img_dir, cui_to_idx, train_transform)
val_dataset = ROCOv2ConceptDataset(val_df, val_img_dir, cui_to_idx, eval_transform)
test_dataset = ROCOv2ConceptDataset(test_df, test_img_dir, cui_to_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True if device.type == 'cuda' else False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True if device.type == 'cuda' else False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True if device.type == 'cuda' else False)

print("DataLoaders initialized successfully.")

DataLoaders initialized successfully.


Class Frequency Computation for Weighted Loss

In [ ]:
num_samples = len(train_df)
class_counts = np.zeros(len(cui_to_idx), dtype=np.float32)

for cuis_str in train_df['CUIs'].dropna():
    for cui in str(cuis_str).split(';'):
        cui = cui.strip()
        if cui in cui_to_idx:
            class_counts[cui_to_idx[cui]] += 1.0

pos_weights = (num_samples - class_counts) / (class_counts + 1e-5)
pos_weights = np.clip(pos_weights, a_min=1.0, a_max=10.0)
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)

print("Class weights calculated and clipped to range [1.0, 10.0].")

Class weights calculated and clipped to range [1.0, 10.0].


Helper Functions for Model Training and Validation

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    total_samples = len(dataloader.dataset)
    if total_samples == 0:
        return 0.0

    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / total_samples

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    total_samples = len(dataloader.dataset)
    if total_samples == 0:
        return 0.0

    with torch.no_grad():
        for images, targets in dataloader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * images.size(0)
    return running_loss / total_samples

Baseline Training (Standard Unweighted BCE Loss)

In [ ]:
model_baseline = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_baseline.fc = nn.Linear(model_baseline.fc.in_features, len(cui_to_idx))
model_baseline = model_baseline.to(device)

criterion_baseline = nn.BCEWithLogitsLoss()
optimizer_baseline = torch.optim.AdamW(model_baseline.parameters(), lr=1e-4, weight_decay=1e-2)

baseline_path = "/content/drive/MyDrive/Colab Notebooks/best_resnet50_baseline.pth"
best_val_loss = float('inf')

print("--- Running Baseline Model Training ---")
for epoch in range(3):
    train_loss = train_one_epoch(model_baseline, train_loader, criterion_baseline, optimizer_baseline, device)
    val_loss = validate(model_baseline, val_loader, criterion_baseline, device)
    print(f"Epoch [{epoch+1}/3] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_baseline.state_dict(), baseline_path)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 135MB/s]


--- Running Baseline Model Training ---
Epoch [1/3] | Train Loss: 0.0153 | Val Loss: 0.0065
Epoch [2/3] | Train Loss: 0.0060 | Val Loss: 0.0060
Epoch [3/3] | Train Loss: 0.0056 | Val Loss: 0.0058


Baseline Test Evaluation (Standard Threshold = 0.5)

In [ ]:
model_baseline.load_state_dict(torch.load(baseline_path, map_location=device))
model_baseline.eval()

all_probs, all_targets = [], []
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        logits = model_baseline(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets.numpy())

y_probs_base = np.vstack(all_probs)
y_true_base = np.vstack(all_targets)

preds_base = (y_probs_base >= 0.5).astype(float)
base_micro = f1_score(y_true_base, preds_base, average='micro', zero_division=0)
base_macro = f1_score(y_true_base, preds_base, average='macro', zero_division=0)
base_samples = f1_score(y_true_base, preds_base, average='samples', zero_division=0)

print("\n--- BASELINE RESULTS (Threshold = 0.5) ---")
print(f"Micro F1:   {base_micro:.4f}")
print(f"Macro F1:   {base_macro:.4f}")
print(f"Samples F1: {base_samples:.4f}")


--- BASELINE RESULTS (Threshold = 0.5) ---
Micro F1:   0.5489
Macro F1:   0.0086
Samples F1: 0.5772


Improved Training (Weighted BCE Loss + Cosine Scheduler)

In [ ]:
model_weighted = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_weighted.fc = nn.Linear(model_weighted.fc.in_features, len(cui_to_idx))
model_weighted = model_weighted.to(device)

criterion_weighted = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer_weighted = torch.optim.AdamW(model_weighted.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler_weighted = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_weighted, T_max=5)

weighted_path = "/content/drive/MyDrive/Colab Notebooks/best_resnet50_weighted.pth"
best_val_loss = float('inf')

print("--- Running Improved Model Training ---")
for epoch in range(3):
    train_loss = train_one_epoch(model_weighted, train_loader, criterion_weighted, optimizer_weighted, device)
    val_loss = validate(model_weighted, val_loader, criterion_weighted, device)
    scheduler_weighted.step()
    print(f"Epoch [{epoch+1}/3] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_weighted.state_dict(), weighted_path)

--- Running Improved Model Training ---
Epoch [1/3] | Train Loss: 0.0474 | Val Loss: 0.0362
Epoch [2/3] | Train Loss: 0.0324 | Val Loss: 0.0337
Epoch [3/3] | Train Loss: 0.0290 | Val Loss: 0.0326


Improved Test Evaluation with Threshold Grid Search

In [ ]:
model_weighted.load_state_dict(torch.load(weighted_path, map_location=device))
model_weighted.eval()

all_probs, all_targets = [], []
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        logits = model_weighted(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_targets.append(targets.numpy())

y_probs_opt = np.vstack(all_probs)
y_true_opt = np.vstack(all_targets)

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
best_opt_macro, best_opt_thresh = 0.0, 0.5

print(f"\n{'Threshold':<12} | {'Micro F1':<10} | {'Macro F1':<10} | {'Samples F1':<10}")
print("-" * 50)

for t in thresholds:
    preds_t = (y_probs_opt >= t).astype(float)
    micro = f1_score(y_true_opt, preds_t, average='micro', zero_division=0)
    macro = f1_score(y_true_opt, preds_t, average='macro', zero_division=0)
    samples = f1_score(y_true_opt, preds_t, average='samples', zero_division=0)
    if macro > best_opt_macro:
        best_opt_macro = macro
        best_opt_thresh = t
    print(f"{t:<12.2f} | {micro:<10.4f} | {macro:<10.4f} | {samples:<10.4f}")

print("-" * 50)
print(f"Optimal Threshold: {best_opt_thresh:.2f} | Best Macro F1: {best_opt_macro:.4f}")


Threshold    | Micro F1   | Macro F1   | Samples F1
--------------------------------------------------
0.05         | 0.0790     | 0.0277     | 0.0886    
0.10         | 0.1307     | 0.0354     | 0.1406    
0.15         | 0.1801     | 0.0400     | 0.1893    
0.20         | 0.2277     | 0.0402     | 0.2367    
0.25         | 0.2737     | 0.0404     | 0.2838    
0.30         | 0.3175     | 0.0400     | 0.3291    
0.40         | 0.3953     | 0.0368     | 0.4101    
0.50         | 0.4593     | 0.0319     | 0.4763    
--------------------------------------------------
Optimal Threshold: 0.25 | Best Macro F1: 0.0404


Final Results Comparison Summary

In [ ]:
preds_opt_final = (y_probs_opt >= best_opt_thresh).astype(float)

opt_micro = f1_score(y_true_opt, preds_opt_final, average='micro', zero_division=0)
opt_macro = f1_score(y_true_opt, preds_opt_final, average='macro', zero_division=0)
opt_samples = f1_score(y_true_opt, preds_opt_final, average='samples', zero_division=0)

summary_df = pd.DataFrame({
    'Metric': ['Micro F1', 'Macro F1', 'Samples F1'],
    'Baseline (Standard Loss, T=0.5)': [base_micro, base_macro, base_samples],
    f'Improved (Weighted Loss, T={best_opt_thresh})': [opt_micro, opt_macro, opt_samples]
})

print("\n--- FINAL MODEL COMPARISON ---")
print(summary_df.to_string(index=False))


--- FINAL MODEL COMPARISON ---
    Metric  Baseline (Standard Loss, T=0.5)  Improved (Weighted Loss, T=0.25)
  Micro F1                         0.548886                          0.273663
  Macro F1                         0.008650                          0.040386
Samples F1                         0.577216                          0.283817


Evaluation on Manually Annotated 18-Concept Subset

In [ ]:
manual_csv_path = "/content/drive/MyDrive/Colab Notebooks/test_concepts_manual.csv"
manual_df = pd.read_csv(manual_csv_path)
manual_df = align_dataframe(manual_df, test_img_dir)

manual_cuis = set()
for cuis_str in manual_df['CUIs'].dropna():
    for cui in str(cuis_str).split(';'):
        cui = cui.strip()
        if cui in cui_to_idx:
            manual_cuis.add(cui)

selected_indices = [cui_to_idx[cui] for cui in sorted(list(manual_cuis))]
print(f"Evaluating on {len(selected_indices)} manual concepts out of 1947 total.")

y_probs_18 = y_probs_opt[:, selected_indices]

y_true_18 = np.zeros((len(manual_df), len(selected_indices)), dtype=np.float32)
for idx, row in manual_df.iterrows():
    if pd.notna(row['CUIs']):
        cuis = [cui.strip() for cui in str(row['CUIs']).split(';')]
        for cui in cuis:
            if cui in cui_to_idx and cui_to_idx[cui] in selected_indices:
                col_idx = selected_indices.index(cui_to_idx[cui])
                y_true_18[idx, col_idx] = 1.0

preds_18 = (y_probs_18 >= best_opt_thresh).astype(float)

m_micro = f1_score(y_true_18, preds_18, average='micro', zero_division=0)
m_macro = f1_score(y_true_18, preds_18, average='macro', zero_division=0)
m_samples = f1_score(y_true_18, preds_18, average='samples', zero_division=0)

print("\n--- PERFORMANCE ON 18 MANUAL CONCEPTS ---")
print(f"Micro F1:   {m_micro:.4f}")
print(f"Macro F1:   {m_macro:.4f}")
print(f"Samples F1: {m_samples:.4f}")

Evaluating on 18 manual concepts out of 1947 total.

--- PERFORMANCE ON 18 MANUAL CONCEPTS ---
Micro F1:   0.7100
Macro F1:   0.5868
Samples F1: 0.7408


Asymmetric Loss (ASL) Implementation and Model Training

In [ ]:
import torch
import torch.nn as nn

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super(AsymmetricLoss, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y):
        xs_pos = torch.sigmoid(x)
        xs_neg = 1.0 - xs_pos

        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        loss_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        loss_pos = loss_pos * ((1.0 - xs_pos) ** self.gamma_pos)

        loss_neg = (1.0 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss_neg = loss_neg * (xs_pos ** self.gamma_neg)

        loss = loss_pos + loss_neg
        return -loss.sum() / x.size(0)

model_asl = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_asl.fc = nn.Linear(model_asl.fc.in_features, len(cui_to_idx))
model_asl = model_asl.to(device)

criterion_asl = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
optimizer_asl = torch.optim.AdamW(model_asl.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler_asl = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_asl, T_max=5)

asl_path = "/content/drive/MyDrive/Colab Notebooks/best_resnet50_asl.pth"
best_val_loss = float('inf')

print("--- Running Model Training with Asymmetric Loss (ASL) ---")
for epoch in range(3):
    train_loss = train_one_epoch(model_asl, train_loader, criterion_asl, optimizer_asl, device)
    val_loss = validate(model_asl, val_loader, criterion_asl, device)
    scheduler_asl.step()

    print(f"Epoch [{epoch+1}/3] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model_asl.state_dict(), asl_path)

--- Running Model Training with Asymmetric Loss (ASL) ---
Epoch [1/3] | Train Loss: 3.0036 | Val Loss: 2.3662
Epoch [2/3] | Train Loss: 2.1089 | Val Loss: 2.2451
Epoch [3/3] | Train Loss: 1.8977 | Val Loss: 2.2001


ASL Model Test Evaluation and Overall 1947 Concepts Comparison

In [ ]:
model_asl.load_state_dict(torch.load(asl_path, map_location=device))
model_asl.eval()

all_probs_asl, all_targets_asl = [], []
with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        logits = model_asl(images)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs_asl.append(probs)
        all_targets_asl.append(targets.numpy())

y_probs_asl = np.vstack(all_probs_asl)
y_true_asl = np.vstack(all_targets_asl)

thresholds = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
best_asl_macro, best_asl_thresh = 0.0, 0.5

print(f"\n{'Threshold':<12} | {'Micro F1':<10} | {'Macro F1':<10} | {'Samples F1':<10}")
print("-" * 50)

for t in thresholds:
    preds_t = (y_probs_asl >= t).astype(float)
    micro = f1_score(y_true_asl, preds_t, average='micro', zero_division=0)
    macro = f1_score(y_true_asl, preds_t, average='macro', zero_division=0)
    samples = f1_score(y_true_asl, preds_t, average='samples', zero_division=0)
    if macro > best_asl_macro:
        best_asl_macro = macro
        best_asl_thresh = t
    print(f"{t:<12.2f} | {micro:<10.4f} | {macro:<10.4f} | {samples:<10.4f}")

preds_asl_opt = (y_probs_asl >= best_asl_thresh).astype(float)
asl_micro = f1_score(y_true_asl, preds_asl_opt, average='micro', zero_division=0)
asl_macro = f1_score(y_true_asl, preds_asl_opt, average='macro', zero_division=0)
asl_samples = f1_score(y_true_asl, preds_asl_opt, average='samples', zero_division=0)

summary_all_df = pd.DataFrame({
    'Metric': ['Micro F1', 'Macro F1', 'Samples F1'],
    'Baseline (Standard BCE)': [base_micro, base_macro, base_samples],
    f'Weighted BCE (T={best_opt_thresh})': [opt_micro, opt_macro, opt_samples],
    f'ASL (T={best_asl_thresh})': [asl_micro, asl_macro, asl_samples]
})

print("\n--- ALL EXPERIMENTS COMPARISON SUMMARY (1947 CONCEPTS) ---")
print(summary_all_df.to_string(index=False))


Threshold    | Micro F1   | Macro F1   | Samples F1
--------------------------------------------------
0.05         | 0.0040     | 0.0034     | 0.0042    
0.10         | 0.0079     | 0.0050     | 0.0094    
0.15         | 0.0166     | 0.0089     | 0.0202    
0.20         | 0.0344     | 0.0156     | 0.0407    
0.25         | 0.0687     | 0.0261     | 0.0788    
0.30         | 0.1327     | 0.0361     | 0.1463    
0.40         | 0.3812     | 0.0385     | 0.3988    
0.50         | 0.5422     | 0.0215     | 0.5652    
--------------------------------------------------
ASL Optimal Threshold: 0.40 | Best Macro F1: 0.0385


Comparative Evaluation on the 18 Manual Concepts Subset

In [ ]:
manual_csv_path = "/content/drive/MyDrive/Colab Notebooks/test_concepts_manual.csv"
manual_df = pd.read_csv(manual_csv_path)
manual_df = align_dataframe(manual_df, test_img_dir)

manual_cuis = set()
for cuis_str in manual_df['CUIs'].dropna():
    for cui in str(cuis_str).split(';'):
        cui = cui.strip()
        if cui in cui_to_idx:
            manual_cuis.add(cui)

selected_indices = [cui_to_idx[cui] for cui in sorted(list(manual_cuis))]

y_true_18 = np.zeros((len(manual_df), len(selected_indices)), dtype=np.float32)
for idx, row in manual_df.iterrows():
    if pd.notna(row['CUIs']):
        cuis = [cui.strip() for cui in str(row['CUIs']).split(';')]
        for cui in cuis:
            if cui in cui_to_idx and cui_to_idx[cui] in selected_indices:
                col_idx = selected_indices.index(cui_to_idx[cui])
                y_true_18[idx, col_idx] = 1.0

y_probs_base_18 = y_probs_base[:, selected_indices]
preds_base_18 = (y_probs_base_18 >= 0.5).astype(float)
m_micro_base = f1_score(y_true_18, preds_base_18, average='micro', zero_division=0)
m_macro_base = f1_score(y_true_18, preds_base_18, average='macro', zero_division=0)
m_samples_base = f1_score(y_true_18, preds_base_18, average='samples', zero_division=0)

y_probs_opt_18 = y_probs_opt[:, selected_indices]
preds_opt_18 = (y_probs_opt_18 >= best_opt_thresh).astype(float)
m_micro_opt = f1_score(y_true_18, preds_opt_18, average='micro', zero_division=0)
m_macro_opt = f1_score(y_true_18, preds_opt_18, average='macro', zero_division=0)
m_samples_opt = f1_score(y_true_18, preds_opt_18, average='samples', zero_division=0)

y_probs_asl_18 = y_probs_asl[:, selected_indices]
preds_asl_18 = (y_probs_asl_18 >= best_asl_thresh).astype(float)
m_micro_asl = f1_score(y_true_18, preds_asl_18, average='micro', zero_division=0)
m_macro_asl = f1_score(y_true_18, preds_asl_18, average='macro', zero_division=0)
m_samples_asl = f1_score(y_true_18, preds_asl_18, average='samples', zero_division=0)

summary_18_df = pd.DataFrame({
    'Metric': ['Micro F1', 'Macro F1', 'Samples F1'],
    'Baseline 18-Concepts': [m_micro_base, m_macro_base, m_samples_base],
    'Weighted 18-Concepts': [m_micro_opt, m_macro_opt, m_samples_opt],
    'ASL 18-Concepts': [m_micro_asl, m_macro_asl, m_samples_asl]
})

print("\n--- PERFORMANCE ON 18 MANUAL CONCEPTS SUBSET ---")
print(summary_18_df.to_string(index=False))


--- PERFORMANCE ON 18 MANUAL CONCEPTS SUBSET ---
    Metric  Baseline 18-Concepts  Weighted 18-Concepts  ASL 18-Concepts
  Micro F1              0.918345              0.709969         0.761930
  Macro F1              0.797557              0.586797         0.637984
Samples F1              0.920586              0.740769         0.784814
